# 아이돌보미 관련 맘카페 게시글 감정분석

In [123]:
import pandas as pd
import pymongo
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rc('font', family='NanumBarunGothic') # 혹은 다른 설치한 Nanum 폰트 사용
import scipy
import scipy.stats as stats
from konlpy.tag import Okt
okt = Okt()
from mecab import MeCab
mecab = MeCab()

# 데이터 로드 

## 감정 사전 로드

In [124]:
sentiword = pd.read_json('../../dataset/SentiWord_info.json')
sentiword

,word,word_root,polarity
0,(-;,(,1
1,(;_;),(;_;),-1
2,(^^),(^^),1
3,(^-^),(^-^),1
4,(^^*,(,1
...,...,...,...
14849,반신반의하다,반신반의,0
14850,신비롭다,신비,1
14851,아리송하다,아리송,-1
14852,알쏭하다,알쏭하,-1


## 게시글 데이터 로드

In [125]:
baby_care_contents = pd.read_json('../../dataset/baby_care_naver_cafe/baby_care_contents.json')
baby_care_contents.drop('_id',axis=1,inplace=True)
# baby_care_contents['contents']

In [126]:
baby_care_reviews = pd.read_json('../../dataset/baby_care_naver_cafe/baby_care_reaction.json')
baby_care_reviews.drop('_id',axis=1,inplace=True)
# baby_care_reviews['review']

# 데이터 전처리

## 중복값 제거

In [127]:
baby_care_contents.drop_duplicates(keep='first',inplace=True)
baby_care_contents.reset_index(inplace=True)
baby_care_contents.drop('index',axis=1,inplace=True)
baby_care_contents

,title,date,contents,like
0,아이돌보미(시터) 선생님 스타일이 너무 안맞으면 어떻게 하시나요?,2024-08-07 11:10:00,나이가 좀 많으세요\n저희 친정 시댁 어른들보다도 6살 더 많으심....\n그래서 그런지 스타일이 옛날식이고 안맞아요\n애랑 재밌게 노는 방법을 잘 모르시는 느낌....\n그냥 제가 놀아주는게 훨씬 나을거같고\n아이도 저랑 노는걸 훨씬 재밌어하고 ㅠㅠ\n제가 일을 해야하니까 어쩔수없이\n파트타임으로 맡기긴하는데\n하루에 거의 5시간을 시터쌤과 보내는데;;\n적지 않은 돈을 쓰면서 계속 맡기는게 맞는건지\n육아스타일때문에 바꾸면\n또 다른 선생님은 맞으리란 보장도 없고\n새롭게 선생님과 적응해야하고 참 그러네요;;;\n(어린이집은 내년에 보내기로 예정되어있어요),0
1,남편이랑 싸운 다음날 육아,2024-08-07 09:41:00,어제 남편하고 싸우고 나니까\n아침에 애기는 너무 귀여운데 육아하기 싫고\n잠깐 어디 맡기고 혼자있고 싶네요ㅠ\n주변에 맡길가족은 없어요\n친구들한테 털어놓자니 자기얼굴에 침뱉기구\n엄마한테 말하면 속상할거같고\n아기낳기전엔 약간 맘카페에 고민글쓰는거 이해못했는데 사실 ㅎㅎ 왜 그런지 알겠어요\n얼른 기운차리고 이유식 주러 가야죠,1
2,맘님이 애가 어리고 복직을 꼭 해야하는 상황,2024-08-06 13:06:00,"저는 어렸을때부터 애기를 너무 좋아했고 처녀때 조카들도 다 키웠다 할만큼 맨날 놀러가서 언니들 자유부인도 시켜줬어요.\n둘째 낳은지 얼마 안됐는데 좀 키워놓고 일을 시작해야 아이들 학원비라도 벌어야 할것같아요.\n회사로 들어가서 일을 하기엔 경력도 무쓸모, 그 직장생활이 너무 힘들었어서 정말 하기 싫어요 ㅠㅠ\n맘님이 만약에 빨리 복직을 해야해서 사람을 구해야하는데 너무 젊은 사람은 부담스러워할까요?\n전 아이가 너무 좋고 이쁘고 사랑스러워서 체력적으로 힘든 시기에도 너무 잘 육아하거든요…\n제가 젤 행복해하면서 일을 한다면 아이를 돌봐주는 일 하고싶은데 부담스러워할지 아님 좋아할지 감이 안오네요 ㅠㅠ",0
3,필리핀 가사도우미? 신청하신 분 있으세요?,2024-08-06 10:30:00,"서울시에서 신청받고있는 필리핀 가사도우미? 가사관리사?\n정확하게말하면 도우미 관리사도 아니고 아이 돌보미라네요\n청소, 빨래, 밥 이런거 하나도안하고\n오로지 아이 씻기고 밥먹이고 등하원시키고 가사는 아기빨래 정도만 한다는데\n페이는 국내 최저임금이랑 동일.\n우리나라에서도 집안일 다해주고 밖에서 힘들게 청소일하시면서 최저임금 받는 분들 수두룩한데...\n저게뭔지;;;\n그래서 도우미 관리사라고 부르면 안되고\n'돌보미 선생님' 이라고 부르라네요 ㅋㅋㅋ\n혹시 여기도 신청하신 분들 있으세요?\n신청하신분들이 이 사실을 제대로 알고 신청하셨는지 궁금합니다\n뉴스에선 계속 가사관리사라고 했거든요\n사실은 가사관리사가 아니고 돌보미라네요.",0
4,아이돌보미 서비스 신청 결제 관련해서 여쭤봅니다,2024-08-04 12:29:00,"안녕하세요.\n여행 다녀와서 애기, 저, 친정엄마 세명 다 코로나 확진\n판정 받고 집에서 칩거 육아 7일차 입니다ㅠㅠ\n한창 아플때 아이돌보미 서비스는 질병케어도 가능하다고 해서 신청했고, 이제 승인 완료 되었네요.\n제가 육휴중이라 시간제로 해서 신청했고, 시간제-가/나/다 별로 이제 자기 부담금이 다른걸로 알고 있어요.\n국민행복카드를 통해서 결제한다고 알고 있는데\n아이돌보미 서비스 신청 후 돌보미 선생님 오시면\n국민행복카드로 시간제 유형별 해당 금액만 결제되는 시스템인지 궁금합니다.\n주변에서 해본 사람도 없고 해서 해보신 분들\n조언 부탁드립니다🙌🏻",0
...,...,...,...,...
2042,정부지원 아이돌보미 서비스...황당해요,2017-10-25 17:39:00,"★ 잠깐! 게시글 작성 전, 필독 공지! ★\n- 카페규정 : http://cafe.naver.com/imsanbu/28123090 \n- 게시판별 운영 정책 : http://cafe.naver.com/imsanbu/35756864\n\n\n\n지금 맞벌이로 일을 하고는있는데 지인소개로 간곳이고 아기가 아직어리고 어린이집 자리가없어서\n아기 돌보면서 알바처럼 하고 있는데요 아기가 지금16개월\n힘들어서 아이돌보미 서비스 이용해볼려고 적응차 신청을 했어요\n4번 이용했구요...\n저는 당연히 애가 어리니까 적응시키기 위해서 돌보미이모 오셨을떄 집에 두시간정도 같이 있다가 외출했구요..\n근데 오늘 4번째 이용했는데 원래 신청한 시간보다 한시간 집에 돌아왔어요\n근데 갑자기 집에가신대요 엄마오면 무조건집에 간대요...\n어의없어서 가시고나서 센터에 전화해서 물어봤더니\n그 이모가 그랬대요\n집에 점심먹을껏도 마땅치 않고 애엄마가 돌보미시간에 아예 다 나가지않고 집에있다가 나간다..이렇게\n자긴 싫다고..... 너무 황당하네요\n그러면서 다음부턴 우리집안온다고 그랬대요\n네가 첫날 점심은 밥솥에 밥해놨고... 냉장고에 있는거 다 꺼내서 드셔두 되요..이렇게말씀드렸거등요\n그리고 이분은 애랑 가지고온 장난감도 정리안하시고 항상 가셨구요 ㅇ애 밥먹이고 설겆이 그대로 놓고 가셨어요\n저는 원래 돌보미는 온전히 애만 보는건줄알았슴니다만..\n오늘 센터에 전화했을때 이것저것 물어보았는데.... 애와 관련된일만 하는거래요\n그래서 장난감정리와 아기가 먹은 식기류는 설겆이하는게 맞다네요...와...도대체 뭘 잘하셨다고...그런말을 하시는건지 궁금하구요 맞벌이로 되어있는데 엄마가 집에있다는둥... 그런소리를 센터에하고...(저도 바쁜시간 지인이 사정봐줘서... 시간낸건데 아기적응때문에)\n집에또 cctv가 없고해서 정말 돌보미가 어떻게 애랑 놀아주는지도 궁금했구요....\n완전적응하면 애 완전히 맡기고 나갈려고했어요...\n원래 아이돌보미가 이런건가요????\n저는 제가없으면 도대체 뭘할려고 그러나 궁금하기 까지 하더라구요 이런일로 그만둔다고 하니...",2
2043,아이돌보미 시간제보육ㅠ,2017-10-25 02:48:00,"1월에 회사에 복직해요~ 계속 일할건 아니고\n두달정도만 나가면 될것같아요.\n1월되면 아기도 15개월쯤 될것같긴한데\n어린이집은 천천히 보내고싶다는 생각이에요ㅠ\n아기성향이 겁도많고 장소낯가림이 있는것 같아요ㅠㅠ문센가서 다른친구가 친한척해도 불편해하고 제품으로 오거든요ㅠㅠ\n 문제는 두달동안 아기케어인데.. 시어머니가 도와주실것 같긴하지만 하루종일은 힘드실것같아요ㅠㅠ\n남편은 교대근무라 다행이 평일 하루정도는 아기를 온전히 봐줄수 있을것 같아요.\n 이런경우 아이돌보미나 시간제보육 어떤게 괜찮나요??ㅠㅠ 아이돌보미가 더 나을것 같은데 오시는 분이 아기케어를 잘해주실지 불안하기도하고ㅠㅠ 시간제보육을 하자니 낯선장소에 낯선 친구들과 지내는게 너무 걱정되요ㅠㅠ\n이제 돌잔치 몇일남겨두고있는데 1월달일이 벌써 너무 걱정이네요ㅠㅠ\n긴글 읽어주셔서 감사합니다\n ★ 잠깐! 게시글 작성 전, 필독 공지! ★\n- 카페규정 : http://cafe.naver.com/imsanbu/28123090 \n- 게시판별 운영 정책 : http://cafe.naver.com/imsanbu/35756864\n",0
2044,정부지원 아이돌보미 서비스...황당해요,2017-10-25 17:39:00,"★ 잠깐! 게시글 작성 전, 필독 공지! ★\n- 카페규정 : http://cafe.naver.com/imsanbu/28123090 \n- 게시판별 운영 정책 : http://cafe.naver.com/imsanbu/35756864\n\n\n\n지금 맞벌이로 일을 하고는있는데 지인소개로 간곳이고 아기가 아직어리고 어린이집 자리가없어서\n아기 돌보면서 알바처럼 하고 있는데요 아기가 지금16개월\n힘들어서 아이돌보미 서비스 이용해볼려고 적응차 신청을 했어요\n4번 이용했구요...\n저는 당연히 애가 어리니까 적응시키기 위해서 돌보미이모 오셨을떄 집에 두시간정도 같이 있다가 외출했구요..\n근데 오늘 4번째 이용했는데 원래 신청한 시간보다 한시간 집에 돌아왔어요\n근데 갑자기 집에가신대요 엄마오면 무조건집에 간대요...\n어의없어서 가시고나서 센터에 전화해서 물어봤더니\n

## 긍,부정 단어 사전에서 필요없는 단어 제거

In [128]:
sentiword = sentiword.loc[~sentiword['word'].isin(['함께','받다'])]

## 필요없는 데이터 삭제

## 줄바꿈 제거

In [129]:
baby_care_contents['contents'] = baby_care_contents['contents'].str.replace('\n',' ')
baby_care_contents['contents'] = baby_care_contents['contents'].str.replace('★ 잠깐! 게시글 작성 전, 필독 공지! ★ - 카페규정 : http://cafe.naver.com/imsanbu/28123090  - 게시판별 운영 정책 : http://cafe.naver.com/imsanbu/35756864',' ')
baby_care_reviews['review'] = baby_care_reviews['review'].str.replace('\n',' ')

## 게시글, 댓글 데이터 병합

In [130]:
momcafe_contents = pd.concat([baby_care_contents['contents'],baby_care_reviews['review']]).reset_index().drop('index',axis=1)
momcafe_contents = momcafe_contents.rename(columns={0:'contents'})
momcafe_contents

,contents
0,나이가 좀 많으세요 저희 친정 시댁 어른들보다도 6살 더 많으심.... 그래서 그런지 스타일이 옛날식이고 안맞아요 애랑 재밌게 노는 방법을 잘 모르시는 느낌.... 그냥 제가 놀아주는게 훨씬 나을거같고 아이도 저랑 노는걸 훨씬 재밌어하고 ㅠㅠ 제가 일을 해야하니까 어쩔수없이 파트타임으로 맡기긴하는데 하루에 거의 5시간을 시터쌤과 보내는데;; 적지 않은 돈을 쓰면서 계속 맡기는게 맞는건지 육아스타일때문에 바꾸면 또 다른 선생님은 맞으리란 보장도 없고 새롭게 선생님과 적응해야하고 참 그러네요;;; (어린이집은 내년에 보내기로 예정되어있어요)
1,어제 남편하고 싸우고 나니까 아침에 애기는 너무 귀여운데 육아하기 싫고 잠깐 어디 맡기고 혼자있고 싶네요ㅠ 주변에 맡길가족은 없어요 친구들한테 털어놓자니 자기얼굴에 침뱉기구 엄마한테 말하면 속상할거같고 아기낳기전엔 약간 맘카페에 고민글쓰는거 이해못했는데 사실 ㅎㅎ 왜 그런지 알겠어요 얼른 기운차리고 이유식 주러 가야죠
2,"저는 어렸을때부터 애기를 너무 좋아했고 처녀때 조카들도 다 키웠다 할만큼 맨날 놀러가서 언니들 자유부인도 시켜줬어요. 둘째 낳은지 얼마 안됐는데 좀 키워놓고 일을 시작해야 아이들 학원비라도 벌어야 할것같아요. 회사로 들어가서 일을 하기엔 경력도 무쓸모, 그 직장생활이 너무 힘들었어서 정말 하기 싫어요 ㅠㅠ 맘님이 만약에 빨리 복직을 해야해서 사람을 구해야하는데 너무 젊은 사람은 부담스러워할까요? 전 아이가 너무 좋고 이쁘고 사랑스러워서 체력적으로 힘든 시기에도 너무 잘 육아하거든요… 제가 젤 행복해하면서 일을 한다면 아이를 돌봐주는 일 하고싶은데 부담스러워할지 아님 좋아할지 감이 안오네요 ㅠㅠ"
3,"서울시에서 신청받고있는 필리핀 가사도우미? 가사관리사? 정확하게말하면 도우미 관리사도 아니고 아이 돌보미라네요 청소, 빨래, 밥 이런거 하나도안하고 오로지 아이 씻기고 밥먹이고 등하원시키고 가사는 아기빨래 정도만 한다는데 페이는 국내 최저임금이랑 동일. 우리나라에서도 집안일 다해주고 밖에서 힘들게 청소일하시면서 최저임금 받는 분들 수두룩한데... 저게뭔지;;; 그래서 도우미 관리사라고 부르면 안되고 '돌보미 선생님' 이라고 부르라네요 ㅋㅋㅋ 혹시 여기도 신청하신 분들 있으세요? 신청하신분들이 이 사실을 제대로 알고 신청하셨는지 궁금합니다 뉴스에선 계속 가사관리사라고 했거든요 사실은 가사관리사가 아니고 돌보미라네요."
4,"안녕하세요. 여행 다녀와서 애기, 저, 친정엄마 세명 다 코로나 확진 판정 받고 집에서 칩거 육아 7일차 입니다ㅠㅠ 한창 아플때 아이돌보미 서비스는 질병케어도 가능하다고 해서 신청했고, 이제 승인 완료 되었네요. 제가 육휴중이라 시간제로 해서 신청했고, 시간제-가/나/다 별로 이제 자기 부담금이 다른걸로 알고 있어요. 국민행복카드를 통해서 결제한다고 알고 있는데 아이돌보미 서비스 신청 후 돌보미 선생님 오시면 국민행복카드로 시간제 유형별 해당 금액만 결제되는 시스템인지 궁금합니다. 주변에서 해본 사람도 없고 해서 해보신 분들 조언 부탁드립니다🙌🏻"
...,...
29801,"아이돌봄 2년동안 3명정도 도우미와 지내봤는데~ 그런분 없었어요. 그 도우미는 아마 어디든 가도 문제가 될 소지가 많네요..3명중 1명만 정리정돈 안하고 애들도 잘 못보시고,,,사실 몇번만 봐도 딱 나와요 애들을 잘 봐주시는 분인지 아닌지가...나머지 2명의 도우미는 모두 만족스러웠었는데..만족도 조사며 센터에 건의며..아마 추후에 불이익은 그 분일거에요~다른분으로 배정받으시면 됩니다^^"
29802,"저도 전혀 cctv 없어도 밑고 맡길만큼 잘해주시는데, 애기 젖병이랑 얼집갈 준비도 다해주시고이유식도 알아서 담아서 가시고, 밥도 도시락 싸서오세요. 그분 나쁜분이네요. 본인이랑 맞는사람 찾을때까지 바꾸어보세요."
29803,허...그 이모님이 너무하셨네요. 저는 10월부터 백일 막지난 아기 맡기고 출근하고있는데요. 정말깔끔하게 잘정리해 놓으세요. 밥도 부담된다고 싸가지고 오시는데... 정말 많이 다르네요.
29804,차로15분정도 거리에 있긴하더라구요~ 한번씩 가보는것도 좋겠네요!!^^ 떨어지는 연습이ㅠㅠ 왜뭔가 속상한걸까요ㅠㅠ


In [131]:
momcafe_contents.drop_duplicates(keep='first',inplace=True)
momcafe_contents.reset_index(inplace=True)
momcafe_contents.drop('index',axis=1,inplace=True)
momcafe_contents

,contents
0,나이가 좀 많으세요 저희 친정 시댁 어른들보다도 6살 더 많으심.... 그래서 그런지 스타일이 옛날식이고 안맞아요 애랑 재밌게 노는 방법을 잘 모르시는 느낌.... 그냥 제가 놀아주는게 훨씬 나을거같고 아이도 저랑 노는걸 훨씬 재밌어하고 ㅠㅠ 제가 일을 해야하니까 어쩔수없이 파트타임으로 맡기긴하는데 하루에 거의 5시간을 시터쌤과 보내는데;; 적지 않은 돈을 쓰면서 계속 맡기는게 맞는건지 육아스타일때문에 바꾸면 또 다른 선생님은 맞으리란 보장도 없고 새롭게 선생님과 적응해야하고 참 그러네요;;; (어린이집은 내년에 보내기로 예정되어있어요)
1,어제 남편하고 싸우고 나니까 아침에 애기는 너무 귀여운데 육아하기 싫고 잠깐 어디 맡기고 혼자있고 싶네요ㅠ 주변에 맡길가족은 없어요 친구들한테 털어놓자니 자기얼굴에 침뱉기구 엄마한테 말하면 속상할거같고 아기낳기전엔 약간 맘카페에 고민글쓰는거 이해못했는데 사실 ㅎㅎ 왜 그런지 알겠어요 얼른 기운차리고 이유식 주러 가야죠
2,"저는 어렸을때부터 애기를 너무 좋아했고 처녀때 조카들도 다 키웠다 할만큼 맨날 놀러가서 언니들 자유부인도 시켜줬어요. 둘째 낳은지 얼마 안됐는데 좀 키워놓고 일을 시작해야 아이들 학원비라도 벌어야 할것같아요. 회사로 들어가서 일을 하기엔 경력도 무쓸모, 그 직장생활이 너무 힘들었어서 정말 하기 싫어요 ㅠㅠ 맘님이 만약에 빨리 복직을 해야해서 사람을 구해야하는데 너무 젊은 사람은 부담스러워할까요? 전 아이가 너무 좋고 이쁘고 사랑스러워서 체력적으로 힘든 시기에도 너무 잘 육아하거든요… 제가 젤 행복해하면서 일을 한다면 아이를 돌봐주는 일 하고싶은데 부담스러워할지 아님 좋아할지 감이 안오네요 ㅠㅠ"
3,"서울시에서 신청받고있는 필리핀 가사도우미? 가사관리사? 정확하게말하면 도우미 관리사도 아니고 아이 돌보미라네요 청소, 빨래, 밥 이런거 하나도안하고 오로지 아이 씻기고 밥먹이고 등하원시키고 가사는 아기빨래 정도만 한다는데 페이는 국내 최저임금이랑 동일. 우리나라에서도 집안일 다해주고 밖에서 힘들게 청소일하시면서 최저임금 받는 분들 수두룩한데... 저게뭔지;;; 그래서 도우미 관리사라고 부르면 안되고 '돌보미 선생님' 이라고 부르라네요 ㅋㅋㅋ 혹시 여기도 신청하신 분들 있으세요? 신청하신분들이 이 사실을 제대로 알고 신청하셨는지 궁금합니다 뉴스에선 계속 가사관리사라고 했거든요 사실은 가사관리사가 아니고 돌보미라네요."
4,"안녕하세요. 여행 다녀와서 애기, 저, 친정엄마 세명 다 코로나 확진 판정 받고 집에서 칩거 육아 7일차 입니다ㅠㅠ 한창 아플때 아이돌보미 서비스는 질병케어도 가능하다고 해서 신청했고, 이제 승인 완료 되었네요. 제가 육휴중이라 시간제로 해서 신청했고, 시간제-가/나/다 별로 이제 자기 부담금이 다른걸로 알고 있어요. 국민행복카드를 통해서 결제한다고 알고 있는데 아이돌보미 서비스 신청 후 돌보미 선생님 오시면 국민행복카드로 시간제 유형별 해당 금액만 결제되는 시스템인지 궁금합니다. 주변에서 해본 사람도 없고 해서 해보신 분들 조언 부탁드립니다🙌🏻"
...,...
15617,"아이돌봄 2년동안 3명정도 도우미와 지내봤는데~ 그런분 없었어요. 그 도우미는 아마 어디든 가도 문제가 될 소지가 많네요..3명중 1명만 정리정돈 안하고 애들도 잘 못보시고,,,사실 몇번만 봐도 딱 나와요 애들을 잘 봐주시는 분인지 아닌지가...나머지 2명의 도우미는 모두 만족스러웠었는데..만족도 조사며 센터에 건의며..아마 추후에 불이익은 그 분일거에요~다른분으로 배정받으시면 됩니다^^"
15618,"저도 전혀 cctv 없어도 밑고 맡길만큼 잘해주시는데, 애기 젖병이랑 얼집갈 준비도 다해주시고이유식도 알아서 담아서 가시고, 밥도 도시락 싸서오세요. 그분 나쁜분이네요. 본인이랑 맞는사람 찾을때까지 바꾸어보세요."
15619,허...그 이모님이 너무하셨네요. 저는 10월부터 백일 막지난 아기 맡기고 출근하고있는데요. 정말깔끔하게 잘정리해 놓으세요. 밥도 부담된다고 싸가지고 오시는데... 정말 많이 다르네요.
15620,차로15분정도 거리에 있긴하더라구요~ 한번씩 가보는것도 좋겠네요!!^^ 떨어지는 연습이ㅠㅠ 왜뭔가 속상한걸까요ㅠㅠ


## 형태소 분리

In [132]:
from konlpy.tag import Okt
okt = Okt()
def Tokenizer(raw):
    word_list = []
    for word, tag in okt.pos(raw,norm=True,stem=True):
        word_list.append(word)
    return " ".join(word_list)

In [133]:
momcafe_contents['contents'] = momcafe_contents['contents'].astype(str)

In [134]:
momcafe_contents['contents_clean'] = momcafe_contents['contents'].apply(Tokenizer)
momcafe_contents

,contents,contents_clean
0,나이가 좀 많으세요 저희 친정 시댁 어른들보다도 6살 더 많으심.... 그래서 그런지 스타일이 옛날식이고 안맞아요 애랑 재밌게 노는 방법을 잘 모르시는 느낌.... 그냥 제가 놀아주는게 훨씬 나을거같고 아이도 저랑 노는걸 훨씬 재밌어하고 ㅠㅠ 제가 일을 해야하니까 어쩔수없이 파트타임으로 맡기긴하는데 하루에 거의 5시간을 시터쌤과 보내는데;; 적지 않은 돈을 쓰면서 계속 맡기는게 맞는건지 육아스타일때문에 바꾸면 또 다른 선생님은 맞으리란 보장도 없고 새롭게 선생님과 적응해야하고 참 그러네요;;; (어린이집은 내년에 보내기로 예정되어있어요),나이 가 좀 많다 저희 친정 시댁 어른 들 보다도 6 살 더 많다 심 .... 그래서 그렇다 스타일 이 옛날 식 이고 안 맞다 애 랑 재밌다 노 는 방법 을 자다 모르다 느낌 .... 그냥 제 가 놀다 훨씬 나다 같다 아이 도 저 랑 놓다 훨씬 재밌다 ㅠㅠ 제 가 일 을 하다 어쩔 수없이 파트타임 으로 맡다 하다 하루 에 거의 5시간 을 시 터쌤 과 보내다 ;; 적지 않다 돈 을 쓰다 계속 맡기다 맞다 건지다 육아 스타일 때문 에 바꾸다 또 다른 선생님 은 맞다 보장 도 없다 새롭다 선생님 과 적응하다 차다 그렇다 ;;; ( 어린이집 은 내년 에 보내다 예정 되어다 )
1,어제 남편하고 싸우고 나니까 아침에 애기는 너무 귀여운데 육아하기 싫고 잠깐 어디 맡기고 혼자있고 싶네요ㅠ 주변에 맡길가족은 없어요 친구들한테 털어놓자니 자기얼굴에 침뱉기구 엄마한테 말하면 속상할거같고 아기낳기전엔 약간 맘카페에 고민글쓰는거 이해못했는데 사실 ㅎㅎ 왜 그런지 알겠어요 얼른 기운차리고 이유식 주러 가야죠,어제 남편 하고 싸우다 나 니까 아침 에 애기 는 너무 귀엽다 육아 하다 싫다 잠깐 어디 맡다 혼자 있다 싶다 ㅠ 주변 에 맡기다 가족 은 없다 친구 들 한테 털다 자다 자기 얼굴 에 침 뱉다 엄마 한테 말 하다 속상하다 같다 아기 낳다 약간 맘 카페 에 고민 글 쓰다 이해 못 하다 사실 ㅎㅎ 왜 그렇다 알다 얼른 기운 차리다 이유식 주다 가야 죠
2,"저는 어렸을때부터 애기를 너무 좋아했고 처녀때 조카들도 다 키웠다 할만큼 맨날 놀러가서 언니들 자유부인도 시켜줬어요. 둘째 낳은지 얼마 안됐는데 좀 키워놓고 일을 시작해야 아이들 학원비라도 벌어야 할것같아요. 회사로 들어가서 일을 하기엔 경력도 무쓸모, 그 직장생활이 너무 힘들었어서 정말 하기 싫어요 ㅠㅠ 맘님이 만약에 빨리 복직을 해야해서 사람을 구해야하는데 너무 젊은 사람은 부담스러워할까요? 전 아이가 너무 좋고 이쁘고 사랑스러워서 체력적으로 힘든 시기에도 너무 잘 육아하거든요… 제가 젤 행복해하면서 일을 한다면 아이를 돌봐주는 일 하고싶은데 부담스러워할지 아님 좋아할지 감이 안오네요 ㅠㅠ","저 는 어리다 때 부터 애기 를 너무 좋아하다 처녀 때 조카 들 도 다 키우다 하다 맨날 놀다 가다 언니 들 자유 부인 도 시키다 주다 . 둘째 낳다 얼마 안 돼다 좀 키우다 일 을 시작 하다 아이 들 학원 비 라도 벌다 하다 같다 . 회사 로 들어가다 일 을 하다 경력 도 무 쓸모 , 그 직장 생활 이 너무 힘들다 정말 하다 싫다 ㅠㅠ 맘 님 이 만약 에 빨리 복직 을 하다 하다 사람 을 구 하다 너무 젊다 사람 은 부담스럽다 하다 요 ? 전 아이 가 너무 좋다 이쁘다 사랑스럽다 체력 적 으로 힘드다 시기 에도 너무 자다 육아 하 거들다 … 제 가 젤 행복하다 일 을 하다 아이 를 돌보다 일 하다 부담스럽다 하다 아니다 좋아하다 감 이 알다 ㅠㅠ"
3,"서울시에서 신청받고있는 필리핀 가사도우미? 가사관리사? 정확하게말하면 도우미 관리사도 아니고 아이 돌보미라네요 청소, 빨래, 밥 이런거 하나도안하고 오로지 아이 씻기고 밥먹이고 등하원시키고 가사는 아기빨래 정도만 한다는데 페이는 국내 최저임금이랑 동일. 우리나라에서도 집안일 다해주고 밖에서 힘들게 청소일하시면서 최저임금 받는 분들 수두룩한데... 저게뭔지;;; 그래서 도우미 관리사라고 부르면 안되고 '돌보미 선생님' 이라고 부르라네요 ㅋㅋㅋ 혹시 여기도 신청하신 분들 있으세요? 신청하신분들이 이 사실을 제대로 알고 신청하셨는지 궁금합니다 뉴스에선 계속 가사관리사라고 했거든요 사실은 가사관리사가 아니고 돌보미라네요.","서울시 에서 신청 받다 필리핀 가사 도우미 ? 가사 관리사 ? 정확하다 도우미 관리사 도 아니다 아이 돌보다 미라 네 요 청소 , 빨래 , 밥 이렇다 하나 도안 하고 오로지 아이 씻다 밥 먹이 고 등 하원 시키다 가사 는 아기 빨래 정도 만 하다 페이 는 국내 최저임금 이랑 동일 . 우리나라 에서도 집안일 다해 주다 밖 에서 힘들다 청소 이다 최저임금 받다 분들 수두 룩 한 데 ... 저 게 무엇 인지 ;;; 그래서 도우미 관리사 라고 부르다 안되다 ' 돌 보미 선생님 ' 이라고 부르다 요 ㅋㅋㅋ 혹시 여기 도 신청 하다 분들 있다 ? 신청 하 신분 들 이 이 사실 을 제대로 알 고 신청 하다 궁금하다 뉴스 에선 계속 가사 관리사 라고 하다 사실 은 가사 관리사 가 아니다 돌보다 미라 네 요 ."
4,"안녕하세요. 여행 다녀와서 애기, 저, 친정엄마 세명 다 코로나 확진 판정 받고 집에서 칩거 육아 7일차 입니다ㅠㅠ 한창 아플때 아이돌보미 서비스는 질병케어도 가능하다고 해서 신청했고, 이제 승인 완료 되었네요. 제가 육휴중이라 시간제로 해서 신청했고, 시간제-가/나/다 별로 이제 자기 부담금이 다른걸로 알고 있어요. 국민행복카드를 통해서 결제한다고 알고 있는데 아이돌보미 서비스 신청 후 돌보미 선생님 오시면 국민행복카드로 시간제 유형별 해당 금액만 결제되는 시스템인지 궁금합니다. 주변에서 해본 사람도 없고 해서 해보신 분들 조언 부탁드립니다🙌🏻","안녕하다 . 여행 다녀오다 애기 , 저 , 친정엄마 세명 다 코로나 확진 판정 받다 집 에서 칩거 육아 7일 차 이다 ㅠㅠ 한창 아프다 때 아이돌 보미 서비스 는 질병 케어 도 가능하다 하다 신청 하다 , 이제 승인 완료 되어다 . 제 가 육 휴 중이 라 시간제 로 하다 신청 하다 , 시간제 - 가다 / 나 / 다 별로 이제 자기 부담 금 이 다른 것 으로 알 고 있다 . 국민 행복 카드 를 통해 서 결제 하다 알 고 있다 아이돌 보미 서비스 신청 후 돌 보미 선생님 오시 면 국민 행복 카드 로 시간제 유형 별 해당 금액 만 결제 되다 시스템 인지 궁금하다 . 주변 에서 해보다 사람 도 없다 하다 해보다 분들 조언 부탁드리다 🙌🏻"
...,...,...
15617,"아이돌봄 2년동안 3명정도 도우미와 지내봤는데~ 그런분 없었어요. 그 도우미는 아마 어디든 가도 문제가 될 소지가 많네요..3명중 1명만 정리정돈 안하고 애들도 잘 못보시고,,,사실 몇번만 봐도 딱 나와요 애들을 잘 봐주시는 분인지 아닌지가...나머지 2명의 도우미는 모두 만족스러웠었는데..만족도 조사며 센터에 건의며..아마 추후에 불이익은 그 분일거에요~다른분으로 배정받으시면 됩니다^^","아이돌 봄 2년 동안 3 명정 도 도우미 와 지내다 보다 ~ 그런 분 없다 . 그 도우미 는 아마 어디 든 가도 문제 가 되다 소지 가 많다 .. 3 명중 1 명 만 정리정돈 안 하고 애 들 도 자다 못 보다 ,,, 사실 몇번 만 보다 딱 나오다 애 들 을 자다 보다 분 인지 아니다 ... 나머지 2 명의 도우미 는 모두 만족스럽다 .. 만족도 조사 며 센터 에 건의 며 .. 아마 추후 에 불이익 은 그 분 이다 ~ 다른 분 으로 배정 받다 되다 ^^"
15618,"저도 전혀 cctv 없어도 밑고 맡길만큼 잘해주시는데, 애기 젖병이랑 얼집갈 준비도 다해주시고이유식도 알아서 담아서 가시고, 밥도 도시락 싸서오세요. 그분 나쁜분이네요. 본인이랑 맞는사람 찾을때까지 바꾸어보세요.","저 도 전혀 cctv 없다 밑 고 맡기다 잘 해주다 , 애기 젖병 이랑 얼집 갈다 준비 도 다해 주시 고 이유식 도 알다 담다 가시 고 , 밥 도 도시락 싸다 . 그 분 나쁘다 불다 . 본인 이랑 맞다 사람 찾다 때 까지 바꾸다 보다 ."
15619,허...그 이모님이 너무하셨네요. 저는 10월부터 백일 막지난 아기 맡기고 출근하고있는데요. 정말깔끔하게 잘정리해 놓으세요. 밥도 부담된다

# 시간제, 종일제 관련 게시글 분류

In [135]:
def count_time(row):
    if row.count('시간제') > row.count('종일제'):
        return '시간제'
    elif row.count('시간제') < row.count('종일제'):
        return '종일제'

In [136]:
momcafe_contents['care_type'] = momcafe_contents['contents'].apply(count_time)
momcafe_contents

,contents,contents_clean,care_type
0,나이가 좀 많으세요 저희 친정 시댁 어른들보다도 6살 더 많으심.... 그래서 그런지 스타일이 옛날식이고 안맞아요 애랑 재밌게 노는 방법을 잘 모르시는 느낌.... 그냥 제가 놀아주는게 훨씬 나을거같고 아이도 저랑 노는걸 훨씬 재밌어하고 ㅠㅠ 제가 일을 해야하니까 어쩔수없이 파트타임으로 맡기긴하는데 하루에 거의 5시간을 시터쌤과 보내는데;; 적지 않은 돈을 쓰면서 계속 맡기는게 맞는건지 육아스타일때문에 바꾸면 또 다른 선생님은 맞으리란 보장도 없고 새롭게 선생님과 적응해야하고 참 그러네요;;; (어린이집은 내년에 보내기로 예정되어있어요),나이 가 좀 많다 저희 친정 시댁 어른 들 보다도 6 살 더 많다 심 .... 그래서 그렇다 스타일 이 옛날 식 이고 안 맞다 애 랑 재밌다 노 는 방법 을 자다 모르다 느낌 .... 그냥 제 가 놀다 훨씬 나다 같다 아이 도 저 랑 놓다 훨씬 재밌다 ㅠㅠ 제 가 일 을 하다 어쩔 수없이 파트타임 으로 맡다 하다 하루 에 거의 5시간 을 시 터쌤 과 보내다 ;; 적지 않다 돈 을 쓰다 계속 맡기다 맞다 건지다 육아 스타일 때문 에 바꾸다 또 다른 선생님 은 맞다 보장 도 없다 새롭다 선생님 과 적응하다 차다 그렇다 ;;; ( 어린이집 은 내년 에 보내다 예정 되어다 ),None
1,어제 남편하고 싸우고 나니까 아침에 애기는 너무 귀여운데 육아하기 싫고 잠깐 어디 맡기고 혼자있고 싶네요ㅠ 주변에 맡길가족은 없어요 친구들한테 털어놓자니 자기얼굴에 침뱉기구 엄마한테 말하면 속상할거같고 아기낳기전엔 약간 맘카페에 고민글쓰는거 이해못했는데 사실 ㅎㅎ 왜 그런지 알겠어요 얼른 기운차리고 이유식 주러 가야죠,어제 남편 하고 싸우다 나 니까 아침 에 애기 는 너무 귀엽다 육아 하다 싫다 잠깐 어디 맡다 혼자 있다 싶다 ㅠ 주변 에 맡기다 가족 은 없다 친구 들 한테 털다 자다 자기 얼굴 에 침 뱉다 엄마 한테 말 하다 속상하다 같다 아기 낳다 약간 맘 카페 에 고민 글 쓰다 이해 못 하다 사실 ㅎㅎ 왜 그렇다 알다 얼른 기운 차리다 이유식 주다 가야 죠,None
2,"저는 어렸을때부터 애기를 너무 좋아했고 처녀때 조카들도 다 키웠다 할만큼 맨날 놀러가서 언니들 자유부인도 시켜줬어요. 둘째 낳은지 얼마 안됐는데 좀 키워놓고 일을 시작해야 아이들 학원비라도 벌어야 할것같아요. 회사로 들어가서 일을 하기엔 경력도 무쓸모, 그 직장생활이 너무 힘들었어서 정말 하기 싫어요 ㅠㅠ 맘님이 만약에 빨리 복직을 해야해서 사람을 구해야하는데 너무 젊은 사람은 부담스러워할까요? 전 아이가 너무 좋고 이쁘고 사랑스러워서 체력적으로 힘든 시기에도 너무 잘 육아하거든요… 제가 젤 행복해하면서 일을 한다면 아이를 돌봐주는 일 하고싶은데 부담스러워할지 아님 좋아할지 감이 안오네요 ㅠㅠ","저 는 어리다 때 부터 애기 를 너무 좋아하다 처녀 때 조카 들 도 다 키우다 하다 맨날 놀다 가다 언니 들 자유 부인 도 시키다 주다 . 둘째 낳다 얼마 안 돼다 좀 키우다 일 을 시작 하다 아이 들 학원 비 라도 벌다 하다 같다 . 회사 로 들어가다 일 을 하다 경력 도 무 쓸모 , 그 직장 생활 이 너무 힘들다 정말 하다 싫다 ㅠㅠ 맘 님 이 만약 에 빨리 복직 을 하다 하다 사람 을 구 하다 너무 젊다 사람 은 부담스럽다 하다 요 ? 전 아이 가 너무 좋다 이쁘다 사랑스럽다 체력 적 으로 힘드다 시기 에도 너무 자다 육아 하 거들다 … 제 가 젤 행복하다 일 을 하다 아이 를 돌보다 일 하다 부담스럽다 하다 아니다 좋아하다 감 이 알다 ㅠㅠ",None
3,"서울시에서 신청받고있는 필리핀 가사도우미? 가사관리사? 정확하게말하면 도우미 관리사도 아니고 아이 돌보미라네요 청소, 빨래, 밥 이런거 하나도안하고 오로지 아이 씻기고 밥먹이고 등하원시키고 가사는 아기빨래 정도만 한다는데 페이는 국내 최저임금이랑 동일. 우리나라에서도 집안일 다해주고 밖에서 힘들게 청소일하시면서 최저임금 받는 분들 수두룩한데... 저게뭔지;;; 그래서 도우미 관리사라고 부르면 안되고 '돌보미 선생님' 이라고 부르라네요 ㅋㅋㅋ 혹시 여기도 신청하신 분들 있으세요? 신청하신분들이 이 사실을 제대로 알고 신청하셨는지 궁금합니다 뉴스에선 계속 가사관리사라고 했거든요 사실은 가사관리사가 아니고 돌보미라네요.","서울시 에서 신청 받다 필리핀 가사 도우미 ? 가사 관리사 ? 정확하다 도우미 관리사 도 아니다 아이 돌보다 미라 네 요 청소 , 빨래 , 밥 이렇다 하나 도안 하고 오로지 아이 씻다 밥 먹이 고 등 하원 시키다 가사 는 아기 빨래 정도 만 하다 페이 는 국내 최저임금 이랑 동일 . 우리나라 에서도 집안일 다해 주다 밖 에서 힘들다 청소 이다 최저임금 받다 분들 수두 룩 한 데 ... 저 게 무엇 인지 ;;; 그래서 도우미 관리사 라고 부르다 안되다 ' 돌 보미 선생님 ' 이라고 부르다 요 ㅋㅋㅋ 혹시 여기 도 신청 하다 분들 있다 ? 신청 하 신분 들 이 이 사실 을 제대로 알 고 신청 하다 궁금하다 뉴스 에선 계속 가사 관리사 라고 하다 사실 은 가사 관리사 가 아니다 돌보다 미라 네 요 .",None
4,"안녕하세요. 여행 다녀와서 애기, 저, 친정엄마 세명 다 코로나 확진 판정 받고 집에서 칩거 육아 7일차 입니다ㅠㅠ 한창 아플때 아이돌보미 서비스는 질병케어도 가능하다고 해서 신청했고, 이제 승인 완료 되었네요. 제가 육휴중이라 시간제로 해서 신청했고, 시간제-가/나/다 별로 이제 자기 부담금이 다른걸로 알고 있어요. 국민행복카드를 통해서 결제한다고 알고 있는데 아이돌보미 서비스 신청 후 돌보미 선생님 오시면 국민행복카드로 시간제 유형별 해당 금액만 결제되는 시스템인지 궁금합니다. 주변에서 해본 사람도 없고 해서 해보신 분들 조언 부탁드립니다🙌🏻","안녕하다 . 여행 다녀오다 애기 , 저 , 친정엄마 세명 다 코로나 확진 판정 받다 집 에서 칩거 육아 7일 차 이다 ㅠㅠ 한창 아프다 때 아이돌 보미 서비스 는 질병 케어 도 가능하다 하다 신청 하다 , 이제 승인 완료 되어다 . 제 가 육 휴 중이 라 시간제 로 하다 신청 하다 , 시간제 - 가다 / 나 / 다 별로 이제 자기 부담 금 이 다른 것 으로 알 고 있다 . 국민 행복 카드 를 통해 서 결제 하다 알 고 있다 아이돌 보미 서비스 신청 후 돌 보미 선생님 오시 면 국민 행복 카드 로 시간제 유형 별 해당 금액 만 결제 되다 시스템 인지 궁금하다 . 주변 에서 해보다 사람 도 없다 하다 해보다 분들 조언 부탁드리다 🙌🏻",시간제
...,...,...,...
15617,"아이돌봄 2년동안 3명정도 도우미와 지내봤는데~ 그런분 없었어요. 그 도우미는 아마 어디든 가도 문제가 될 소지가 많네요..3명중 1명만 정리정돈 안하고 애들도 잘 못보시고,,,사실 몇번만 봐도 딱 나와요 애들을 잘 봐주시는 분인지 아닌지가...나머지 2명의 도우미는 모두 만족스러웠었는데..만족도 조사며 센터에 건의며..아마 추후에 불이익은 그 분일거에요~다른분으로 배정받으시면 됩니다^^","아이돌 봄 2년 동안 3 명정 도 도우미 와 지내다 보다 ~ 그런 분 없다 . 그 도우미 는 아마 어디 든 가도 문제 가 되다 소지 가 많다 .. 3 명중 1 명 만 정리정돈 안 하고 애 들 도 자다 못 보다 ,,, 사실 몇번 만 보다 딱 나오다 애 들 을 자다 보다 분 인지 아니다 ... 나머지 2 명의 도우미 는 모두 만족스럽다 .. 만족도 조사 며 센터 에 건의 며 .. 아마 추후 에 불이익 은 그 분 이다 ~ 다른 분 으로 배정 받다 되다 ^^",None
15618,"저도 전혀 cctv 없어도 밑고 맡길만큼 잘해주시는데, 애기 젖병이랑 얼집갈 준비도 다해주시고이유식도 알아서 담아서 가시고, 밥도 도시락 싸서오세요. 그분 나쁜분이네요. 본인이랑 맞는사람 찾을때까지 바꾸어보세요.","저 도 전혀 cctv 없다 밑 고 맡기다 잘 해주다 , 애기 젖병 이랑 얼집 갈다 준비 도 다해 주시 고 이유식 도 알다 담다 가시 고 , 밥 도 도시락 싸다 . 그 분 나쁘다 불다 . 본인 이랑 맞다 사람 찾다 때 까지 바꾸다 보다 .",None
15619,허...그 이모님이 너무하셨네요. 저는 10월부터 백

In [137]:
momcafe_contents['care_type'].value_counts()

care_type
시간제    260
종일제    114
Name: count, dtype: int64

In [138]:
parttime_contents = momcafe_contents.loc[momcafe_contents['care_type']=='시간제']
alltime_contents = momcafe_contents.loc[momcafe_contents['care_type']=='종일제']

# 긍, 부정 분류

In [139]:
def get_morphs(raw):
    pos = list(sentiword.loc[sentiword['polarity']==1]['word'])
    neg = list(sentiword.loc[sentiword['polarity']==-1]['word'])
    raw_list = raw.split(' ')
    pos_list = []
    neg_list = []
    for morph in raw_list:
        if morph in pos:
            pos_list.append(morph)
        elif morph in neg:
            neg_list.append(morph)
    if len(pos_list) > len(neg_list):
        return pos_list
    elif len(pos_list) < len(neg_list):
        return neg_list
    else:
        return []

def get_score(raw):
    pos = list(sentiword.loc[sentiword['polarity']==1]['word'])
    neg = list(sentiword.loc[sentiword['polarity']==-1]['word'])
    raw_list = raw.split(' ')
    pos_list = []
    neg_list = []
    for morph in raw_list:
        if morph in pos:
            pos_list.append(morph)
        elif morph in neg:
            neg_list.append(morph)
    if len(pos_list) > len(neg_list):
        return len(pos_list)
    elif len(pos_list) < len(neg_list):
        return -len(neg_list)
    else:
        return 0

    return len(raw)

def get_label(raw):
    if raw > 0 :
        label = "긍정"
    elif raw < 0 :
        label = "부정"
    else:
        label = "중립"
    
    return label

In [140]:
parttime_contents['words_list'] = parttime_contents['contents_clean'].apply(get_morphs)
parttime_contents['score'] = parttime_contents['contents_clean'].apply(get_score)
parttime_contents['label'] = parttime_contents['score'].apply(get_label)
alltime_contents['words_list'] = alltime_contents['contents_clean'].apply(get_morphs)
alltime_contents['score'] = alltime_contents['contents_clean'].apply(get_score)
alltime_contents['label'] = alltime_contents['score'].apply(get_label)

/tmp/ipykernel_695/1477840231.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parttime_contents['words_list'] = parttime_contents['contents_clean'].apply(get_morphs)
/tmp/ipykernel_695/1477840231.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parttime_contents['score'] = parttime_contents['contents_clean'].apply(get_score)
/tmp/ipykernel_695/1477840231.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value in

In [141]:
parttime_contents['label'] = parttime_contents['score'].apply(get_label)
alltime_contents['label'] = alltime_contents['score'].apply(get_label)

/tmp/ipykernel_695/1877602452.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parttime_contents['label'] = parttime_contents['score'].apply(get_label)
/tmp/ipykernel_695/1877602452.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  alltime_contents['label'] = alltime_contents['score'].apply(get_label)


In [142]:
parttime_contents['label'].value_counts()

label
중립    108
부정     78
긍정     74
Name: count, dtype: int64

In [143]:
alltime_contents['label'].value_counts()

label
중립    50
부정    38
긍정    26
Name: count, dtype: int64

# 불안에 관련된 게시글/댓글

In [170]:
confuse_contents = momcafe_contents.loc[momcafe_contents['contents'].str.contains('불안')]
confuse_contents

,contents,contents_clean,care_type
16,"저희 아가는 7~8개월때부터 얼집넣어서 현재 28개월 여아인데 매우 잘 적응해서 다니고 있다고 생각했는데.., 제 착각였나봐요.., 오늘 아이가 너무 안가려고 계속 잠든채로 옷은 입히면 벗고, 입히면 벗고.., 요 녀석이 잠든척 하면서 안가려고 시간끄는구나 싶긴했어요.., 오늘따라 이렇게 실랑이하느라 늦어서 8시 반에 등원시켰는데.., (이시간에도 등원한 아이들이 없더라구요ㅠ) 마침 담임선생님 출근해계셔서 얘기를 잠깐 나눴는데.., 이번주 요몇일.., 아침에 진정이 안될정도로 울면서 엄마, 아빠를 불렀어요.., 보통 몇시에 자나요? 물어보셔서.., 11시에서 12시 사이에 잠든다니까ㅠㅠ 애가 피곤하기도 하고, 컨디션이 안좋아서 그런거 같다고.., 그리고 이젠 제법 커서.., 본인이 8시에 오면 친구가 없다는것도 알고.., 다들 일찍 가는데.., 마지막에 가니까 ㅠㅠ 이걸 좀 아는거 같다고.., 혹시 등하원 주변에 도움주실분 없냐고.., ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ 제 출근시간이 9시.., 회사까지 자차이용 50분.., ㅠ이라.., 거의 8시까지 부랴부랴 애 깨우고.., 아님 잠든채로 옷입혀서 유모차태워 얼집에 등원시켰다가.., 1년전까지는 그래도 시부모님이 가까이 계셔서 하원시간에 좀 일찍 데려오셨거든요.., 근데 시부모님이 이사가셔서ㅠㅠ 제가 월급여 조정해서 4시까지 근무로 변경했는데 4시에 끝나자마자 집에 달려와도.., 5시 ㅠㅠ 이거저거 챙겨서 하원시키러 가면 5시반 그때가면 저희 애가 거의 꼴찌거나 꼴찌에서 두번째 ㅠㅠ 남편은 더 바쁘고 정신없어서 시간적으로 빼기는 더 어렵고 ㅠ 남편에게 얘기하니.., 우리아파트는 구축이라 다들 전업맘이 많아서 그렇다고.., 옆동네 신도시 가면 워킹맘 많아서 다들 우리가 등하원하는 시간에 많이들 등하원시킬꺼라는데..,ㅠㅠ 돈벌어서 옆에 신도시로 이사갈 계획을 가지고 있긴 한데.., 남편 말이 맞나?? 괜히 나 그만두는거 싫으니까 저딴소리하나?? 이런 생각도 드네요;;ㄷㄷㄷ 더 좋은곳으로 가려면 돈을 벌어야 하는데 돈을 벌면 아이가 심적으로 불안정해질수도 있는거 같아.., 요즘 1차 멘붕 겪고있어요;;; 저도 회사를 계속 다녀야하나.., 생각해봤을때 어떤 초등학생 엄마는.., 초등학교 가도 저학년은 12시반.., 고학년은.., 2시반 3시에 애들 끝나서 시간이 애매해 엄마가 일하기 쉽지 않다고 얘기하시는데.., 엄마가 일하면서 애 키우는게 정말 쉽지는 않은거 같네요.., 저도 육휴후 1년만에 복직한 회사에서 감떨어졌네.., 일처리 속도도 늦어지고, 밥먹듯 지각하고, 회사에서는 싫은소리도 많이 듣다보니.., 자존감떨어지고ㅠ 눈치도 많이 보여서.., 요즘 정말 ㅠㅠ 스트레스가 너무 높아지는 시기인거 같아요.., 너무 스트레스받아 여기다 끄적이고 하소연해봐요.., 이건 그냥 지나가는 잠깐의 고비인지? 아님 빨리 퇴사하고, 육아하면서 개인사업으로 살방법을 찾아야할지? 워킹맘들은 아기 28갤에서 3-4살까지 어떻게 양육하셨는지 궁금해요..,","저희 아가 는 7~8 개월때 부터 얼집 넣다 현재 28 개월 이다 매우 자다 적응하다 다니다 있다 생각 하다 .., 제 착각 이다 보다 .., 오늘 아이 가 너무 알다 계속 잠들다 채 로 옷 은 입히다 벗다 , 입히다 벗다 .., 요 녀석 이 잠들다 척 하다 알다 시간 끄다 싶다 하다 .., 오늘 따르다 이렇게 실랑이 하다 늦다 8시 반 에 등원 시키다 .., ( 이 시간 에도 등원 한 아이 들 이 없다 ㅠ ) 마침 담임 선생님 출근 하다 얘기 를 잠깐 나누다 .., 이번 주 요 몇 일 .., 아침 에 진정 이 안되다 정도 로 울면 서 엄마 , 아빠 를 부르다 .., 보통 몇 시 에 자다 ? 물어보다 .., 11시 에서 12시 사이 에 잠들다 ㅠㅠ 애가 피곤하다 하다 , 컨디션 이 안좋다 그렇다 같다 .., 그리고 이 젠 제법 커서 .., 본인 이 8시 에 오다 친구 가 없다 알 고 .., 다 들다 일찍 가다 .., 마지막 에 가다 ㅠㅠ 이 걸 좀 알다 같다 .., 혹시 등 하원 주변 에 도움 주 실 분 없다 .., ㅠㅠㅠ ㅠㅠㅠ 제 출근시간 이 9시 .., 회사 까지 자 차이다 50분 .., ㅠ 이르다 .., 거의 8시 까지 부랴부랴 애 깨우다 .., 아니다 잠들다 채 로 옷 입히다 유모차 태우다 얼집 에 등원 시키다 .., 1년 전까지는 그래도 시부모 님 이 가까이 계시다 하원시 간 에 좀 일찍 데려오다 .., 근데 시부모 님 이 이사 갈다 ㅠㅠ 제 가 월급 여 조정 하다 4시 까지 근무 로 변경 하다 4시 에 끝나다 집 에 달려오다 .., 5시 ㅠㅠ 이 거저 거 챙기다 하원 시키다 가면 5시 반 그때 가면 저희 애가 거의 꼴 찌다 꼴찌 에서 두번째 ㅠㅠ 남편 은 더 바쁘다 정신 없다 시간 적 으로 빼기 는 더 어렵다 ㅠ 남편 에게 얘기 하다 .., 우리 아파트 는 구축 이라 다 들다 전업 맘 이 많다 그렇다고 .., 옆 동네 신도시 가면 워킹맘 많다 다 들다 우리 가 등 하 원하다 시간 에 많이 들다 등 하원 시키다 .., ㅠㅠ 돈 벌다 옆 에 신도시 로 이사 갈다 계획 을 가지 고 있다 한데 .., 남편 말 이 맞다 ?? 괜히 나 그만두다 싫다 저딴 소 리하나 ?? 이렇다 생각 도 드네 요 ;; ㄷㄷㄷ 더 좋다 곳 으로 가다 돈 을 벌다 하다 돈 을 벌다 아이 가 심 적 으로 불안정하다 수도 있다 같다 .., 요즘 1 차 멘붕 겪다 ;;; 저 도 회사 를 계속 다니다 .., 생각 해봤다 때 어떻다 초등학생 엄마 는 .., 초등학교 가도 저학년 은 12시 반 .., 고학년 은 .., 2시 반 3시 에 애 들 끝나다 시간 이 애매하다 엄마 가 일 하다 쉬다 않다 얘기 하다 .., 엄마 가 일 하다 애 키우다 정말 쉬다 않다 같다 .., 저 도 육 휴 후 1년 만에 복직 한 회사 에서 감 떨어지다 .., 일 처리 속도 도 늦어지다 , 밥 먹듯 지각 하고 , 회사 에서는 싫다 소리 도 많이 듣다 보다 .., 자존감 떨어지다 ㅠ 눈치 도 많이 보 여서 .., 요즘 정말 ㅠㅠ 스트레스 가 너무 높아지다 시기 인거 같다 .., 너무 스트레스 받다 여기 다 끄적 이고 하소연 해보다 .., 이 것 은 그냥 지나가다 잠깐 의 고비 인지 ? 아니다 빨리 퇴사 하고 , 육아 하다 개인 사업 으로 살 방법 을 찾다 야하다 ? 워킹맘 들 은 아기 28 개다 3-4 살 까지 어떻다 양육 하다 궁금하다 ..,",None
21,여아 두돌 아기이구요 파트타임 선생님이 계세요 이전에 시터분들이 몇번 그만두시고... 이분을 모셨는데 같은 아파트 같은동이라서 오래 일하시길 바라며 한달 넘게 모시고 있어요 정부지원 아이돌보미라서 드리는 비용은 정해져있습니다. 제가 재택근무라서 갑자기 선생님이 그만두면 상당히 곤란해요 제발 오래 계시면 좋겠는데..... 이전에 갑자기 그만두신분들도 있어서 약간 불안함을 안고 사는 기분이 있긴해요 ㅠㅠㅠㅠㅠㅠ 최대한 친절하게 웃으며 호의적으로 대해드리고 있습니다.... 지금 시터나 돌보미 쓰고 계시다면 얼마나 오래하셨나요? 혹시 오래 근무하게 하시는 노하우가 있을까요....?,여 아 두 돌 아기 이구 요 파트타임 선생님 이 계세 요 이전 에 시 터 분들 이 몇번 그만두다 ... 이분 을 모시다 같다 아파트 같다 동이 라서 오래 일 하다 바라다 한 달 넘다 모시 고 있다 정부 지원 아이돌 보미 라서 드리다 비용 은 정해지다 . 제 가 재택근무 라서 갑자기 선생님 이 그만두다 상당하다 곤란하다 제발 오래 계시 면 좋다 ..... 이전 에 갑자기 그만 두 신분 들 도 있다 약간 불안하다 안고 살다 기분 이 있다 해 요 ㅠㅠㅠ 최대한 친절하다 웃다 호의 적 으로 대해 드리다 있다 .... 지금 시 터 나 돌 보미 쓰다 계시다 얼마나 오래 하다 ? 혹시 오래 근무 하다 하다 노하우 가 있다 ....?,None
28,"혼자 아기보는데 너무 힘들어서.. 돌보미서비스 종종 이용하구있어요 산후이모님도, 돌봄선생님도 제가 집에 있을 때만 와주셨는데 다음달에 일주일정도는 제가 집에 없어서.. 돌봄선생님이랑 아기만 있을 거 같아요. 하루 5시간이요. 불안해서 시아버지(70대)

# 긍정 게시글 토픽 분류

In [171]:
stopwords = ['보미','아이','아기','신분','가요','대요','도우미','계속','정도','이모','지금','얘기','동의','애기','애가','아가','걱정','불안','일제','방이','말씀','조언','조금','요즘','제일','부분','생각','서비스','이용']
# stopwords = []
def Tokenizer_noun(raw, pos=["Noun"], stopword=stopwords):
    word_list = []
    for word, tag in okt.pos(raw,norm=True,stem=True):
        if len(word) > 1 and tag in pos and word not in stopword:
            if mecab.pos(word)[0][1] in ["NNG"]:
                word_list.append(word)
    return " ".join(word_list)

In [172]:
def noun_set(raw):
    noun_list = list(set(raw.split(' ')))
    return " ".join(noun_list)

In [173]:
confuse_contents['noun_list'] = confuse_contents['contents'].apply(Tokenizer_noun)

/tmp/ipykernel_695/4029482251.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  confuse_contents['noun_list'] = confuse_contents['contents'].apply(Tokenizer_noun)


In [174]:
confuse_contents['noun_set'] = confuse_contents['noun_list'].apply(noun_set)

/tmp/ipykernel_695/4017758516.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  confuse_contents['noun_set'] = confuse_contents['noun_list'].apply(noun_set)


In [175]:
confuse_contents = confuse_contents.query("noun_list !=''")
confuse_contents = confuse_contents.reset_index()

In [176]:
confuse_contents.drop('index',axis=1,inplace=True)
confuse_contents

,contents,contents_clean,care_type,noun_list,noun_set
0,"저희 아가는 7~8개월때부터 얼집넣어서 현재 28개월 여아인데 매우 잘 적응해서 다니고 있다고 생각했는데.., 제 착각였나봐요.., 오늘 아이가 너무 안가려고 계속 잠든채로 옷은 입히면 벗고, 입히면 벗고.., 요 녀석이 잠든척 하면서 안가려고 시간끄는구나 싶긴했어요.., 오늘따라 이렇게 실랑이하느라 늦어서 8시 반에 등원시켰는데.., (이시간에도 등원한 아이들이 없더라구요ㅠ) 마침 담임선생님 출근해계셔서 얘기를 잠깐 나눴는데.., 이번주 요몇일.., 아침에 진정이 안될정도로 울면서 엄마, 아빠를 불렀어요.., 보통 몇시에 자나요? 물어보셔서.., 11시에서 12시 사이에 잠든다니까ㅠㅠ 애가 피곤하기도 하고, 컨디션이 안좋아서 그런거 같다고.., 그리고 이젠 제법 커서.., 본인이 8시에 오면 친구가 없다는것도 알고.., 다들 일찍 가는데.., 마지막에 가니까 ㅠㅠ 이걸 좀 아는거 같다고.., 혹시 등하원 주변에 도움주실분 없냐고.., ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ 제 출근시간이 9시.., 회사까지 자차이용 50분.., ㅠ이라.., 거의 8시까지 부랴부랴 애 깨우고.., 아님 잠든채로 옷입혀서 유모차태워 얼집에 등원시켰다가.., 1년전까지는 그래도 시부모님이 가까이 계셔서 하원시간에 좀 일찍 데려오셨거든요.., 근데 시부모님이 이사가셔서ㅠㅠ 제가 월급여 조정해서 4시까지 근무로 변경했는데 4시에 끝나자마자 집에 달려와도.., 5시 ㅠㅠ 이거저거 챙겨서 하원시키러 가면 5시반 그때가면 저희 애가 거의 꼴찌거나 꼴찌에서 두번째 ㅠㅠ 남편은 더 바쁘고 정신없어서 시간적으로 빼기는 더 어렵고 ㅠ 남편에게 얘기하니.., 우리아파트는 구축이라 다들 전업맘이 많아서 그렇다고.., 옆동네 신도시 가면 워킹맘 많아서 다들 우리가 등하원하는 시간에 많이들 등하원시킬꺼라는데..,ㅠㅠ 돈벌어서 옆에 신도시로 이사갈 계획을 가지고 있긴 한데.., 남편 말이 맞나?? 괜히 나 그만두는거 싫으니까 저딴소리하나?? 이런 생각도 드네요;;ㄷㄷㄷ 더 좋은곳으로 가려면 돈을 벌어야 하는데 돈을 벌면 아이가 심적으로 불안정해질수도 있는거 같아.., 요즘 1차 멘붕 겪고있어요;;; 저도 회사를 계속 다녀야하나.., 생각해봤을때 어떤 초등학생 엄마는.., 초등학교 가도 저학년은 12시반.., 고학년은.., 2시반 3시에 애들 끝나서 시간이 애매해 엄마가 일하기 쉽지 않다고 얘기하시는데.., 엄마가 일하면서 애 키우는게 정말 쉽지는 않은거 같네요.., 저도 육휴후 1년만에 복직한 회사에서 감떨어졌네.., 일처리 속도도 늦어지고, 밥먹듯 지각하고, 회사에서는 싫은소리도 많이 듣다보니.., 자존감떨어지고ㅠ 눈치도 많이 보여서.., 요즘 정말 ㅠㅠ 스트레스가 너무 높아지는 시기인거 같아요.., 너무 스트레스받아 여기다 끄적이고 하소연해봐요.., 이건 그냥 지나가는 잠깐의 고비인지? 아님 빨리 퇴사하고, 육아하면서 개인사업으로 살방법을 찾아야할지? 워킹맘들은 아기 28갤에서 3-4살까지 어떻게 양육하셨는지 궁금해요..,","저희 아가 는 7~8 개월때 부터 얼집 넣다 현재 28 개월 이다 매우 자다 적응하다 다니다 있다 생각 하다 .., 제 착각 이다 보다 .., 오늘 아이 가 너무 알다 계속 잠들다 채 로 옷 은 입히다 벗다 , 입히다 벗다 .., 요 녀석 이 잠들다 척 하다 알다 시간 끄다 싶다 하다 .., 오늘 따르다 이렇게 실랑이 하다 늦다 8시 반 에 등원 시키다 .., ( 이 시간 에도 등원 한 아이 들 이 없다 ㅠ ) 마침 담임 선생님 출근 하다 얘기 를 잠깐 나누다 .., 이번 주 요 몇 일 .., 아침 에 진정 이 안되다 정도 로 울면 서 엄마 , 아빠 를 부르다 .., 보통 몇 시 에 자다 ? 물어보다 .., 11시 에서 12시 사이 에 잠들다 ㅠㅠ 애가 피곤하다 하다 , 컨디션 이 안좋다 그렇다 같다 .., 그리고 이 젠 제법 커서 .., 본인 이 8시 에 오다 친구 가 없다 알 고 .., 다 들다 일찍 가다 .., 마지막 에 가다 ㅠㅠ 이 걸 좀 알다 같다 .., 혹시 등 하원 주변 에 도움 주 실 분 없다 .., ㅠㅠㅠ ㅠㅠㅠ 제 출근시간 이 9시 .., 회사 까지 자 차이다 50분 .., ㅠ 이르다 .., 거의 8시 까지 부랴부랴 애 깨우다 .., 아니다 잠들다 채 로 옷 입히다 유모차 태우다 얼집 에 등원 시키다 .., 1년 전까지는 그래도 시부모 님 이 가까이 계시다 하원시 간 에 좀 일찍 데려오다 .., 근데 시부모 님 이 이사 갈다 ㅠㅠ 제 가 월급 여 조정 하다 4시 까지 근무 로 변경 하다 4시 에 끝나다 집 에 달려오다 .., 5시 ㅠㅠ 이 거저 거 챙기다 하원 시키다 가면 5시 반 그때 가면 저희 애가 거의 꼴 찌다 꼴찌 에서 두번째 ㅠㅠ 남편 은 더 바쁘다 정신 없다 시간 적 으로 빼기 는 더 어렵다 ㅠ 남편 에게 얘기 하다 .., 우리 아파트 는 구축 이라 다 들다 전업 맘 이 많다 그렇다고 .., 옆 동네 신도시 가면 워킹맘 많다 다 들다 우리 가 등 하 원하다 시간 에 많이 들다 등 하원 시키다 .., ㅠㅠ 돈 벌다 옆 에 신도시 로 이사 갈다 계획 을 가지 고 있다 한데 .., 남편 말 이 맞다 ?? 괜히 나 그만두다 싫다 저딴 소 리하나 ?? 이렇다 생각 도 드네 요 ;; ㄷㄷㄷ 더 좋다 곳 으로 가다 돈 을 벌다 하다 돈 을 벌다 아이 가 심 적 으로 불안정하다 수도 있다 같다 .., 요즘 1 차 멘붕 겪다 ;;; 저 도 회사 를 계속 다니다 .., 생각 해봤다 때 어떻다 초등학생 엄마 는 .., 초등학교 가도 저학년 은 12시 반 .., 고학년 은 .., 2시 반 3시 에 애 들 끝나다 시간 이 애매하다 엄마 가 일 하다 쉬다 않다 얘기 하다 .., 엄마 가 일 하다 애 키우다 정말 쉬다 않다 같다 .., 저 도 육 휴 후 1년 만에 복직 한 회사 에서 감 떨어지다 .., 일 처리 속도 도 늦어지다 , 밥 먹듯 지각 하고 , 회사 에서는 싫다 소리 도 많이 듣다 보다 .., 자존감 떨어지다 ㅠ 눈치 도 많이 보 여서 .., 요즘 정말 ㅠㅠ 스트레스 가 너무 높아지다 시기 인거 같다 .., 너무 스트레스 받다 여기 다 끄적 이고 하소연 해보다 .., 이 것 은 그냥 지나가다 잠깐 의 고비 인지 ? 아니다 빨리 퇴사 하고 , 육아 하다 개인 사업 으로 살 방법 을 찾다 야하다 ? 워킹맘 들 은 아기 28 개다 3-4 살 까지 어떻다 양육 하다 궁금하다 ..,",None,현재 착각 오늘 시간 오늘 실랑이 등원 시간 등원 담임 선생님 출근 이번 아침 진정 울면 엄마 아빠 보통 사이 컨디션 본인 친구 마지막 하원 주변 도움 출근시간 회사 유모차 등원 시부모 시부모 이사 월급 조정 근무 변경 거저 하원 그때 꼴찌 남편 정신 시간 남편 아파트 구축 전업 동네 워킹맘 시간 하원 이사 계획 가지 남편 리하나 수도 회사 초등학생 엄마 초등학교 고학년 시간 엄마 엄마 복직 회사 처리 속도 지각 회사 소리 자존감 눈치 스트레스 시기 스트레스 고비 퇴사 육아 개인 사업 방법 워킹맘 양육,거저 속도 수도 눈치 아빠 고비 꼴찌 실랑이 주변 고학년 진정 워킹맘 오늘 시간 변경 사업 하원 전업 구축 초등학교 현재 리하나 지각 엄마 남편 월급 출근 시부모 스트레스 초등학생 그때 개인 사이 이번 처리 보통 출근시간 복직 조정 양육 컨디션 동네 가지 육아 유모차 시기 본인 마지막 아파트 소리 등원 착각 울면 방법 선생님 담임 계획 회사 근무 퇴사 도움 자존감 친구 아침 정신 이사
1,여아 두돌 아기이구요 파트타임 선생님이 계세요 이전에 시터분들이 몇번 그만두시고... 이분을 모셨는데 같은 아파트 같은동이라서 오래 일하시길 바라며 한달 넘게 모시고 있어요 정부지원 아이돌보미라서 드리는 비용은 정해져있습니다. 제가 재택근무라서 갑자기 선생님이 그만두면 상당히 곤란해요 제발 오래 계시면 좋겠는데..... 이전에 갑자기 그만두신분들도 있어서 약간 불안함을 안고 사는 기분이 있긴해요 ㅠㅠㅠㅠㅠㅠ 최대한 친절하게 웃으며 호의적으로 대해드리고 있습니다.... 지금 시터나 돌보미 쓰고 계시다면 얼마나 오래하셨나요? 혹시 오래 근무하게 하

In [177]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidfVectorizer_topic = TfidfVectorizer(use_idf=True,max_df=0.8,min_df=0.02,)
features = tfidfVectorizer_topic.fit_transform(confuse_contents['noun_list'])
features.toarray()[:2]
dictionary_list = tfidfVectorizer_topic.get_feature_names_out()


In [178]:
from sklearn.decomposition import LatentDirichletAllocation
lda_model = LatentDirichletAllocation(n_components=3, random_state=36) #인스턴스화 #n_components 토픽의 갯수
lda_model.fit(features)
import pandas as pd
pd.set_option('display.max_colwidth', None)
## 상위 단어 추출 
## 0 확률 1은 dictionary
topics_list = list()
for topic in lda_model.components_:
    df_datas = [topic, dictionary_list]
    df_topics = pd.DataFrame(data=df_datas)
    df_topics= df_topics.T
    df_topics = df_topics.sort_values(0, ascending=False)
    # print(df_topics[:3])
    topics_text = ' '.join(df_topics[1].values[:20])# 시리즈 형식으로 출력 get values from series / index 
    topics_list.append(topics_text)
topics_list_add = [['Topic1', 'Topic2','Topic3'],topics_list]
df_topics_keywords = pd.DataFrame(topics_list_add)
df_topics_keywords=df_topics_keywords.T
df_topics_keywords

,0,1
0,Topic1,선생님 남편 시간 마음 사람 고민 어린이집 지원 엄마 정부 산후 혼자 처음 도움 센터 가정 신랑 하루 복직 케어
1,Topic2,신청 설치 사랑 티비 관계 엄마 선생님 라면 불안감 애착 입장 고용 현실 감사 사람 아침 댓글 출산휴가 정서 스트레스
2,Topic3,어린이집 조리 친정 병원 엄마 뉴스 표현 상황 보호자 복직 시작 회사 추천 내년 직장 육아 시댁 느낌 분리 출산


In [179]:
list_topics = []
for i in range(len(lda_model.components_)):
    df_datas = [lda_model.components_[i], dictionary_list]
    df_topics = pd.DataFrame(data=df_datas).T
    df_topics = df_topics.dropna()
    df_topics = df_topics.sort_values(0, ascending=False).reset_index()
    df_topics.rename(columns = {1 : i+1}, inplace = True)
    df_topics.rename(columns = {0 : 'score'}, inplace = True)
    list_topics.append(df_topics.loc[:4,['score',i+1]])
df_topic = pd.concat(list_topics,axis=1)
df_topic



,score,1,score,2,score,3
0,10.92945,선생님,4.268043,신청,7.170739,어린이집
1,7.636795,남편,3.972702,설치,3.37089,조리
2,5.948173,시간,3.839706,사랑,3.127502,친정
3,5.776771,마음,3.670828,티비,3.035075,병원
4,5.549606,사람,3.488358,관계,2.946753,엄마


In [180]:
import pyLDAvis
import pyLDAvis.lda_model
vis = pyLDAvis.lda_model.prepare(lda_model,features,tfidfVectorizer_topic)
vis
pyLDAvis.enable_notebook()
components_display = pyLDAvis.display(vis)
components_display

In [181]:
topics_list = list()
for topic in lda_model.components_:
    df_datas = [topic, dictionary_list]
    df_topics = pd.DataFrame(data=df_datas)
    df_topics= df_topics.T
    df_topics = df_topics.sort_values(0, ascending=False)

In [182]:
topics_output = lda_model.transform(features)
df_topics_score = pd.DataFrame(topics_output)
df_topics_score['dominant_topic_number']=np.argmax(topics_output, axis=1)
df_topics_score['dominant_topic_number'].value_counts()

dominant_topic_number
0    101
1     47
2     41
Name: count, dtype: int64

In [183]:
confuse_contents.loc[:,'content_topic'] = df_topics_score['dominant_topic_number']
confuse_contents

,contents,contents_clean,care_type,noun_list,noun_set,content_topic
0,"저희 아가는 7~8개월때부터 얼집넣어서 현재 28개월 여아인데 매우 잘 적응해서 다니고 있다고 생각했는데.., 제 착각였나봐요.., 오늘 아이가 너무 안가려고 계속 잠든채로 옷은 입히면 벗고, 입히면 벗고.., 요 녀석이 잠든척 하면서 안가려고 시간끄는구나 싶긴했어요.., 오늘따라 이렇게 실랑이하느라 늦어서 8시 반에 등원시켰는데.., (이시간에도 등원한 아이들이 없더라구요ㅠ) 마침 담임선생님 출근해계셔서 얘기를 잠깐 나눴는데.., 이번주 요몇일.., 아침에 진정이 안될정도로 울면서 엄마, 아빠를 불렀어요.., 보통 몇시에 자나요? 물어보셔서.., 11시에서 12시 사이에 잠든다니까ㅠㅠ 애가 피곤하기도 하고, 컨디션이 안좋아서 그런거 같다고.., 그리고 이젠 제법 커서.., 본인이 8시에 오면 친구가 없다는것도 알고.., 다들 일찍 가는데.., 마지막에 가니까 ㅠㅠ 이걸 좀 아는거 같다고.., 혹시 등하원 주변에 도움주실분 없냐고.., ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ 제 출근시간이 9시.., 회사까지 자차이용 50분.., ㅠ이라.., 거의 8시까지 부랴부랴 애 깨우고.., 아님 잠든채로 옷입혀서 유모차태워 얼집에 등원시켰다가.., 1년전까지는 그래도 시부모님이 가까이 계셔서 하원시간에 좀 일찍 데려오셨거든요.., 근데 시부모님이 이사가셔서ㅠㅠ 제가 월급여 조정해서 4시까지 근무로 변경했는데 4시에 끝나자마자 집에 달려와도.., 5시 ㅠㅠ 이거저거 챙겨서 하원시키러 가면 5시반 그때가면 저희 애가 거의 꼴찌거나 꼴찌에서 두번째 ㅠㅠ 남편은 더 바쁘고 정신없어서 시간적으로 빼기는 더 어렵고 ㅠ 남편에게 얘기하니.., 우리아파트는 구축이라 다들 전업맘이 많아서 그렇다고.., 옆동네 신도시 가면 워킹맘 많아서 다들 우리가 등하원하는 시간에 많이들 등하원시킬꺼라는데..,ㅠㅠ 돈벌어서 옆에 신도시로 이사갈 계획을 가지고 있긴 한데.., 남편 말이 맞나?? 괜히 나 그만두는거 싫으니까 저딴소리하나?? 이런 생각도 드네요;;ㄷㄷㄷ 더 좋은곳으로 가려면 돈을 벌어야 하는데 돈을 벌면 아이가 심적으로 불안정해질수도 있는거 같아.., 요즘 1차 멘붕 겪고있어요;;; 저도 회사를 계속 다녀야하나.., 생각해봤을때 어떤 초등학생 엄마는.., 초등학교 가도 저학년은 12시반.., 고학년은.., 2시반 3시에 애들 끝나서 시간이 애매해 엄마가 일하기 쉽지 않다고 얘기하시는데.., 엄마가 일하면서 애 키우는게 정말 쉽지는 않은거 같네요.., 저도 육휴후 1년만에 복직한 회사에서 감떨어졌네.., 일처리 속도도 늦어지고, 밥먹듯 지각하고, 회사에서는 싫은소리도 많이 듣다보니.., 자존감떨어지고ㅠ 눈치도 많이 보여서.., 요즘 정말 ㅠㅠ 스트레스가 너무 높아지는 시기인거 같아요.., 너무 스트레스받아 여기다 끄적이고 하소연해봐요.., 이건 그냥 지나가는 잠깐의 고비인지? 아님 빨리 퇴사하고, 육아하면서 개인사업으로 살방법을 찾아야할지? 워킹맘들은 아기 28갤에서 3-4살까지 어떻게 양육하셨는지 궁금해요..,","저희 아가 는 7~8 개월때 부터 얼집 넣다 현재 28 개월 이다 매우 자다 적응하다 다니다 있다 생각 하다 .., 제 착각 이다 보다 .., 오늘 아이 가 너무 알다 계속 잠들다 채 로 옷 은 입히다 벗다 , 입히다 벗다 .., 요 녀석 이 잠들다 척 하다 알다 시간 끄다 싶다 하다 .., 오늘 따르다 이렇게 실랑이 하다 늦다 8시 반 에 등원 시키다 .., ( 이 시간 에도 등원 한 아이 들 이 없다 ㅠ ) 마침 담임 선생님 출근 하다 얘기 를 잠깐 나누다 .., 이번 주 요 몇 일 .., 아침 에 진정 이 안되다 정도 로 울면 서 엄마 , 아빠 를 부르다 .., 보통 몇 시 에 자다 ? 물어보다 .., 11시 에서 12시 사이 에 잠들다 ㅠㅠ 애가 피곤하다 하다 , 컨디션 이 안좋다 그렇다 같다 .., 그리고 이 젠 제법 커서 .., 본인 이 8시 에 오다 친구 가 없다 알 고 .., 다 들다 일찍 가다 .., 마지막 에 가다 ㅠㅠ 이 걸 좀 알다 같다 .., 혹시 등 하원 주변 에 도움 주 실 분 없다 .., ㅠㅠㅠ ㅠㅠㅠ 제 출근시간 이 9시 .., 회사 까지 자 차이다 50분 .., ㅠ 이르다 .., 거의 8시 까지 부랴부랴 애 깨우다 .., 아니다 잠들다 채 로 옷 입히다 유모차 태우다 얼집 에 등원 시키다 .., 1년 전까지는 그래도 시부모 님 이 가까이 계시다 하원시 간 에 좀 일찍 데려오다 .., 근데 시부모 님 이 이사 갈다 ㅠㅠ 제 가 월급 여 조정 하다 4시 까지 근무 로 변경 하다 4시 에 끝나다 집 에 달려오다 .., 5시 ㅠㅠ 이 거저 거 챙기다 하원 시키다 가면 5시 반 그때 가면 저희 애가 거의 꼴 찌다 꼴찌 에서 두번째 ㅠㅠ 남편 은 더 바쁘다 정신 없다 시간 적 으로 빼기 는 더 어렵다 ㅠ 남편 에게 얘기 하다 .., 우리 아파트 는 구축 이라 다 들다 전업 맘 이 많다 그렇다고 .., 옆 동네 신도시 가면 워킹맘 많다 다 들다 우리 가 등 하 원하다 시간 에 많이 들다 등 하원 시키다 .., ㅠㅠ 돈 벌다 옆 에 신도시 로 이사 갈다 계획 을 가지 고 있다 한데 .., 남편 말 이 맞다 ?? 괜히 나 그만두다 싫다 저딴 소 리하나 ?? 이렇다 생각 도 드네 요 ;; ㄷㄷㄷ 더 좋다 곳 으로 가다 돈 을 벌다 하다 돈 을 벌다 아이 가 심 적 으로 불안정하다 수도 있다 같다 .., 요즘 1 차 멘붕 겪다 ;;; 저 도 회사 를 계속 다니다 .., 생각 해봤다 때 어떻다 초등학생 엄마 는 .., 초등학교 가도 저학년 은 12시 반 .., 고학년 은 .., 2시 반 3시 에 애 들 끝나다 시간 이 애매하다 엄마 가 일 하다 쉬다 않다 얘기 하다 .., 엄마 가 일 하다 애 키우다 정말 쉬다 않다 같다 .., 저 도 육 휴 후 1년 만에 복직 한 회사 에서 감 떨어지다 .., 일 처리 속도 도 늦어지다 , 밥 먹듯 지각 하고 , 회사 에서는 싫다 소리 도 많이 듣다 보다 .., 자존감 떨어지다 ㅠ 눈치 도 많이 보 여서 .., 요즘 정말 ㅠㅠ 스트레스 가 너무 높아지다 시기 인거 같다 .., 너무 스트레스 받다 여기 다 끄적 이고 하소연 해보다 .., 이 것 은 그냥 지나가다 잠깐 의 고비 인지 ? 아니다 빨리 퇴사 하고 , 육아 하다 개인 사업 으로 살 방법 을 찾다 야하다 ? 워킹맘 들 은 아기 28 개다 3-4 살 까지 어떻다 양육 하다 궁금하다 ..,",None,현재 착각 오늘 시간 오늘 실랑이 등원 시간 등원 담임 선생님 출근 이번 아침 진정 울면 엄마 아빠 보통 사이 컨디션 본인 친구 마지막 하원 주변 도움 출근시간 회사 유모차 등원 시부모 시부모 이사 월급 조정 근무 변경 거저 하원 그때 꼴찌 남편 정신 시간 남편 아파트 구축 전업 동네 워킹맘 시간 하원 이사 계획 가지 남편 리하나 수도 회사 초등학생 엄마 초등학교 고학년 시간 엄마 엄마 복직 회사 처리 속도 지각 회사 소리 자존감 눈치 스트레스 시기 스트레스 고비 퇴사 육아 개인 사업 방법 워킹맘 양육,거저 속도 수도 눈치 아빠 고비 꼴찌 실랑이 주변 고학년 진정 워킹맘 오늘 시간 변경 사업 하원 전업 구축 초등학교 현재 리하나 지각 엄마 남편 월급 출근 시부모 스트레스 초등학생 그때 개인 사이 이번 처리 보통 출근시간 복직 조정 양육 컨디션 동네 가지 육아 유모차 시기 본인 마지막 아파트 소리 등원 착각 울면 방법 선생님 담임 계획 회사 근무 퇴사 도움 자존감 친구 아침 정신 이사,0
1,여아 두돌 아기이구요 파트타임 선생님이 계세요 이전에 시터분들이 몇번 그만두시고... 이분을 모셨는데 같은 아파트 같은동이라서 오래 일하시길 바라며 한달 넘게 모시고 있어요 정부지원 아이돌보미라서 드리는 비용은 정해져있습니다. 제가 재택근무라서 갑자기 선생님이 그만두면 상당히 곤란해요 제발 오래 계시면 좋겠는데..... 이전에 갑자기 그만두신분들도 있어서 약간 불안함을 안고 사는 기분이 있긴해요 ㅠㅠㅠㅠㅠㅠ 최대한 친절하게 웃으며 호의적으로 대해드리고 있습니다.... 지금 시터나 돌보미 쓰고 계시다면 얼마나 오래하셨

In [184]:
confuse_contents.loc[confuse_contents['content_topic']==0]

,contents,contents_clean,care_type,noun_list,noun_set,content_topic
0,"저희 아가는 7~8개월때부터 얼집넣어서 현재 28개월 여아인데 매우 잘 적응해서 다니고 있다고 생각했는데.., 제 착각였나봐요.., 오늘 아이가 너무 안가려고 계속 잠든채로 옷은 입히면 벗고, 입히면 벗고.., 요 녀석이 잠든척 하면서 안가려고 시간끄는구나 싶긴했어요.., 오늘따라 이렇게 실랑이하느라 늦어서 8시 반에 등원시켰는데.., (이시간에도 등원한 아이들이 없더라구요ㅠ) 마침 담임선생님 출근해계셔서 얘기를 잠깐 나눴는데.., 이번주 요몇일.., 아침에 진정이 안될정도로 울면서 엄마, 아빠를 불렀어요.., 보통 몇시에 자나요? 물어보셔서.., 11시에서 12시 사이에 잠든다니까ㅠㅠ 애가 피곤하기도 하고, 컨디션이 안좋아서 그런거 같다고.., 그리고 이젠 제법 커서.., 본인이 8시에 오면 친구가 없다는것도 알고.., 다들 일찍 가는데.., 마지막에 가니까 ㅠㅠ 이걸 좀 아는거 같다고.., 혹시 등하원 주변에 도움주실분 없냐고.., ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ 제 출근시간이 9시.., 회사까지 자차이용 50분.., ㅠ이라.., 거의 8시까지 부랴부랴 애 깨우고.., 아님 잠든채로 옷입혀서 유모차태워 얼집에 등원시켰다가.., 1년전까지는 그래도 시부모님이 가까이 계셔서 하원시간에 좀 일찍 데려오셨거든요.., 근데 시부모님이 이사가셔서ㅠㅠ 제가 월급여 조정해서 4시까지 근무로 변경했는데 4시에 끝나자마자 집에 달려와도.., 5시 ㅠㅠ 이거저거 챙겨서 하원시키러 가면 5시반 그때가면 저희 애가 거의 꼴찌거나 꼴찌에서 두번째 ㅠㅠ 남편은 더 바쁘고 정신없어서 시간적으로 빼기는 더 어렵고 ㅠ 남편에게 얘기하니.., 우리아파트는 구축이라 다들 전업맘이 많아서 그렇다고.., 옆동네 신도시 가면 워킹맘 많아서 다들 우리가 등하원하는 시간에 많이들 등하원시킬꺼라는데..,ㅠㅠ 돈벌어서 옆에 신도시로 이사갈 계획을 가지고 있긴 한데.., 남편 말이 맞나?? 괜히 나 그만두는거 싫으니까 저딴소리하나?? 이런 생각도 드네요;;ㄷㄷㄷ 더 좋은곳으로 가려면 돈을 벌어야 하는데 돈을 벌면 아이가 심적으로 불안정해질수도 있는거 같아.., 요즘 1차 멘붕 겪고있어요;;; 저도 회사를 계속 다녀야하나.., 생각해봤을때 어떤 초등학생 엄마는.., 초등학교 가도 저학년은 12시반.., 고학년은.., 2시반 3시에 애들 끝나서 시간이 애매해 엄마가 일하기 쉽지 않다고 얘기하시는데.., 엄마가 일하면서 애 키우는게 정말 쉽지는 않은거 같네요.., 저도 육휴후 1년만에 복직한 회사에서 감떨어졌네.., 일처리 속도도 늦어지고, 밥먹듯 지각하고, 회사에서는 싫은소리도 많이 듣다보니.., 자존감떨어지고ㅠ 눈치도 많이 보여서.., 요즘 정말 ㅠㅠ 스트레스가 너무 높아지는 시기인거 같아요.., 너무 스트레스받아 여기다 끄적이고 하소연해봐요.., 이건 그냥 지나가는 잠깐의 고비인지? 아님 빨리 퇴사하고, 육아하면서 개인사업으로 살방법을 찾아야할지? 워킹맘들은 아기 28갤에서 3-4살까지 어떻게 양육하셨는지 궁금해요..,","저희 아가 는 7~8 개월때 부터 얼집 넣다 현재 28 개월 이다 매우 자다 적응하다 다니다 있다 생각 하다 .., 제 착각 이다 보다 .., 오늘 아이 가 너무 알다 계속 잠들다 채 로 옷 은 입히다 벗다 , 입히다 벗다 .., 요 녀석 이 잠들다 척 하다 알다 시간 끄다 싶다 하다 .., 오늘 따르다 이렇게 실랑이 하다 늦다 8시 반 에 등원 시키다 .., ( 이 시간 에도 등원 한 아이 들 이 없다 ㅠ ) 마침 담임 선생님 출근 하다 얘기 를 잠깐 나누다 .., 이번 주 요 몇 일 .., 아침 에 진정 이 안되다 정도 로 울면 서 엄마 , 아빠 를 부르다 .., 보통 몇 시 에 자다 ? 물어보다 .., 11시 에서 12시 사이 에 잠들다 ㅠㅠ 애가 피곤하다 하다 , 컨디션 이 안좋다 그렇다 같다 .., 그리고 이 젠 제법 커서 .., 본인 이 8시 에 오다 친구 가 없다 알 고 .., 다 들다 일찍 가다 .., 마지막 에 가다 ㅠㅠ 이 걸 좀 알다 같다 .., 혹시 등 하원 주변 에 도움 주 실 분 없다 .., ㅠㅠㅠ ㅠㅠㅠ 제 출근시간 이 9시 .., 회사 까지 자 차이다 50분 .., ㅠ 이르다 .., 거의 8시 까지 부랴부랴 애 깨우다 .., 아니다 잠들다 채 로 옷 입히다 유모차 태우다 얼집 에 등원 시키다 .., 1년 전까지는 그래도 시부모 님 이 가까이 계시다 하원시 간 에 좀 일찍 데려오다 .., 근데 시부모 님 이 이사 갈다 ㅠㅠ 제 가 월급 여 조정 하다 4시 까지 근무 로 변경 하다 4시 에 끝나다 집 에 달려오다 .., 5시 ㅠㅠ 이 거저 거 챙기다 하원 시키다 가면 5시 반 그때 가면 저희 애가 거의 꼴 찌다 꼴찌 에서 두번째 ㅠㅠ 남편 은 더 바쁘다 정신 없다 시간 적 으로 빼기 는 더 어렵다 ㅠ 남편 에게 얘기 하다 .., 우리 아파트 는 구축 이라 다 들다 전업 맘 이 많다 그렇다고 .., 옆 동네 신도시 가면 워킹맘 많다 다 들다 우리 가 등 하 원하다 시간 에 많이 들다 등 하원 시키다 .., ㅠㅠ 돈 벌다 옆 에 신도시 로 이사 갈다 계획 을 가지 고 있다 한데 .., 남편 말 이 맞다 ?? 괜히 나 그만두다 싫다 저딴 소 리하나 ?? 이렇다 생각 도 드네 요 ;; ㄷㄷㄷ 더 좋다 곳 으로 가다 돈 을 벌다 하다 돈 을 벌다 아이 가 심 적 으로 불안정하다 수도 있다 같다 .., 요즘 1 차 멘붕 겪다 ;;; 저 도 회사 를 계속 다니다 .., 생각 해봤다 때 어떻다 초등학생 엄마 는 .., 초등학교 가도 저학년 은 12시 반 .., 고학년 은 .., 2시 반 3시 에 애 들 끝나다 시간 이 애매하다 엄마 가 일 하다 쉬다 않다 얘기 하다 .., 엄마 가 일 하다 애 키우다 정말 쉬다 않다 같다 .., 저 도 육 휴 후 1년 만에 복직 한 회사 에서 감 떨어지다 .., 일 처리 속도 도 늦어지다 , 밥 먹듯 지각 하고 , 회사 에서는 싫다 소리 도 많이 듣다 보다 .., 자존감 떨어지다 ㅠ 눈치 도 많이 보 여서 .., 요즘 정말 ㅠㅠ 스트레스 가 너무 높아지다 시기 인거 같다 .., 너무 스트레스 받다 여기 다 끄적 이고 하소연 해보다 .., 이 것 은 그냥 지나가다 잠깐 의 고비 인지 ? 아니다 빨리 퇴사 하고 , 육아 하다 개인 사업 으로 살 방법 을 찾다 야하다 ? 워킹맘 들 은 아기 28 개다 3-4 살 까지 어떻다 양육 하다 궁금하다 ..,",None,현재 착각 오늘 시간 오늘 실랑이 등원 시간 등원 담임 선생님 출근 이번 아침 진정 울면 엄마 아빠 보통 사이 컨디션 본인 친구 마지막 하원 주변 도움 출근시간 회사 유모차 등원 시부모 시부모 이사 월급 조정 근무 변경 거저 하원 그때 꼴찌 남편 정신 시간 남편 아파트 구축 전업 동네 워킹맘 시간 하원 이사 계획 가지 남편 리하나 수도 회사 초등학생 엄마 초등학교 고학년 시간 엄마 엄마 복직 회사 처리 속도 지각 회사 소리 자존감 눈치 스트레스 시기 스트레스 고비 퇴사 육아 개인 사업 방법 워킹맘 양육,거저 속도 수도 눈치 아빠 고비 꼴찌 실랑이 주변 고학년 진정 워킹맘 오늘 시간 변경 사업 하원 전업 구축 초등학교 현재 리하나 지각 엄마 남편 월급 출근 시부모 스트레스 초등학생 그때 개인 사이 이번 처리 보통 출근시간 복직 조정 양육 컨디션 동네 가지 육아 유모차 시기 본인 마지막 아파트 소리 등원 착각 울면 방법 선생님 담임 계획 회사 근무 퇴사 도움 자존감 친구 아침 정신 이사,0
1,여아 두돌 아기이구요 파트타임 선생님이 계세요 이전에 시터분들이 몇번 그만두시고... 이분을 모셨는데 같은 아파트 같은동이라서 오래 일하시길 바라며 한달 넘게 모시고 있어요 정부지원 아이돌보미라서 드리는 비용은 정해져있습니다. 제가 재택근무라서 갑자기 선생님이 그만두면 상당히 곤란해요 제발 오래 계시면 좋겠는데..... 이전에 갑자기 그만두신분들도 있어서 약간 불안함을 안고 사는 기분이 있긴해요 ㅠㅠㅠㅠㅠㅠ 최대한 친절하게 웃으며 호의적으로 대해드리고 있습니다.... 지금 시터나 돌보미 쓰고 계시다면 얼마나 오래하셨

# 결론

## 급하게 아이돌봄 서비스를 이용하거나 짧은 시간 서비스를 이용하고 싶어하는 사람들을 위한 정책 필요
### - 아동돌봄 서비스의 다양화 정책
## 현재 수요에 비해 공급이 부족한 것으로 보임 아이돌봄 서비스 수요와 공급 현황에 대한 분석 필요
### - 공급을 늘릴 수 있는 방안 분석
## 전문성과 직업의식이 부족한 아이돌보미가 많음. 부모들을 안심시킬 수 있도록 아이돌보미에 대한 관리가 필요
### - 아이돌보미를 국가에서 일방적으로 배정해주는 시스템이 아닌 매칭할 수 있는 커뮤니티 제공
### - 아이돌보미와 관련된 자격증, 교육에 대한 정책 강화 필요
